# Numerische Berechnung der Vorwärtskinematik am Beispiel des vereinfachten Scara Roboters

Jonas Frei, 21.09.2026, jonas.frei@ost.ch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

## Initialisierung

**Robotergeometrie**

In [ ]:
l1 = 1.5
l2 = 1

**Zielpose**

In [ ]:
p = np.array([1.0, 2.0])
if np.sqrt(p[0] ** 2 + p[1] ** 2) > l1 + l2:
    raise ValueError('Zielposition ausserhalb der Reichweite des Roboters!')

**Vorwärtskinematik**

In [ ]:
def kin(q):
    return np.array([
        l1 * np.cos(q[0]) + l2 * np.cos(q[0] + q[1]),
        l1 * np.sin(q[0]) + l2 * np.sin(q[0] + q[1])
    ])

**Inverse Jacobi-Matrix**

In [ ]:
def Jinv(q):
    return np.array([
        [np.cos(q[0] + q[1]) / (l1 * np.sin(q[1])),
         np.sin(q[0] + q[1]) / (l1 * np.sin(q[1]))],
        [-(l2 * np.cos(q[0] + q[1]) + l1 * np.cos(q[0])) / (l1 * l2 * np.sin(q[1])),
         -(l2 * np.sin(q[0] + q[1]) + l1 * np.sin(q[0])) / (l1 * l2 * np.sin(q[1]))]
    ])

**Abbruchkriterien**

In [ ]:
RHO = 1e-4
MAXITER = 50

**Schrittweite**

In [ ]:
LAMBDA = 0.5

## Newton-Raphson Algorithmus

**Schätzen der Gelenkswinkel**, Berechnen der Endeffektorpose, des Schätzfehlers und der neuen Schätzung, bis das Abbruchkriterium erreicht ist

In [ ]:
qest = np.array([-np.pi, np.pi + 0.5])

if qest[1] % np.pi < 1e-4:
    raise ValueError('Roboter nahe an singulärer Stellung! Inverse Jakobimatrix kann nicht berechnet werden!')
else:
    i = 0
    while i < MAXITER:
        # Berechnen der Endeffektorpose aus den geschätzten oder gemessenen Gelenkswinkeln
        pest = kin(qest)

        # Berechnen des Schätzfehlers in allen kartesischen Koordinaten
        deltap = p - pest

        # Berechnen der inversen Jacobi-Matrix aus den geschätzten Gelenkwinkeln
        # und berechnen des Schätzfehlers der Gelenkwinkel
        if qest[1] % np.pi < 1e-4:
            raise ValueError('Roboter nahe an singulärer Stellung! Inverse Jakobimatrix kann nicht berechnet werden!')
        deltaq = Jinv(qest) @ deltap

        # Neue Schätzung der Gelenkwinkel
        qest = qest + LAMBDA * deltaq

        # Abbruch, falls Schätzfehler in allen kartesischen Koordinaten kleiner als
        # Abbruchkriterium oder maximale Anzahl Iterationen erreicht worden sind
        if np.all(np.abs(deltap) < RHO):
            break
        i += 1

    if i == MAXITER:
        warnings.warn("Maximale Anzahl Iterationen erreicht, Ergebnis ist möglicherweise ungenau!")

    q = qest
    p = kin(q)

## Ausgabe

In [ ]:
print(f"q = ({q[0]}, {q[1]})")
print(f"p = ({p[0]}, {p[1]})")
print(f"Iterationen: {i}")

## Plotten

In [ ]:
x_pts = np.array([0, l1 * np.cos(q[0]), p[0]])
y_pts = np.array([0, l1 * np.sin(q[0]), p[1]])

fig, ax = plt.subplots()
ax.plot(x_pts, y_pts, linewidth=5)

labels = ['{0},{1}', '{2}', '{TCP}']
for xi, yi, label in zip(x_pts + 0.05, y_pts - 0.05, labels):
    ax.text(xi, yi, label)

ax.set_title('Vereinfachter Scara Roboter')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal', adjustable='box')
ax.grid(True)
ax.set_xlim(1.2 * np.array([-l1 - l2, l1 + l2]))
ax.set_ylim(1.2 * np.array([-l1 - l2, l1 + l2]))

plt.show()